# Visualization
In the previous chapters you have already seen some mechanisms to display geometry in a jupyter notebook, such
as Shapely and Matplotlib. In this chapter we'll look at more advanded visualization libraries:

* [Folium](#Folium)
* [ipyleaflet](#ipyleaflet)
* [Bokeh](#Bokeh)
* [pydeck](#pydeck)


## Folium
Whenever you visit a website that has some kind of interactive map, the map
functionality is likely provided with a javascript library 
such as [Leaflet](http://leafletjs.com), [OpenLayers](https://openlayers.org),
[maplibre](https://maplibre.org/), or [google maps](https://geemap.org/).

The Python module [Folium](https://github.com/python-visualization/folium) makes
it possible to visualize data that has been manipulated in Python on an
interactive Leaflet map in a jupyter notebook or python based website.

### Basics
We will start with the most minimal map using the default OpenStreetMap base map.
See [Folium Quickstart](https://python-visualization.github.io/folium/quickstart.html).


In [1]:
import folium

folium.Map(location=[45.75, 21.25], zoom_start=12)

The `location` keyword argument (there are many more, with sensible defaults) provides the Map centre
in Latitude and Longitude (Northing, Easting), `zoom_start` indicates the initial zoom level.


### GeoJSON overlay
It gets interesting when you can overlay the map with data manipulated
via Python. Here we overlay the map with the Polygons of all countries.


In [2]:
countries = f'../data/countries.json'

folium_map = folium.Map(
    location=[0, 0],
    zoom_start=2  
)

folium.GeoJson(
    countries,
    name='countries'
).add_to(folium_map)

folium.LayerControl().add_to(folium_map)

display(folium_map)

### Folium and Streamlit
Just like with Juptyer, Folium can also be combined with python web frameworks such as Streamlit
[Streamlit](https://streamlit.io) is a platform to create interactive web apps for your python data scripts.
See for example this [Folium+Streamlit example](https://folium.streamlit.app/). 



## ipyleaflet
[ipyleaflet](https://ipyleaflet.readthedocs.io) provides similar functionality as folium, however because
it is based on [ipywidgets](https://ipywidgets.readthedocs.io), it integrates with other components from
the ipywidgets ecosystem (sliders, datagrids, tabs).

Links:

* GitHub: https://github.com/jupyter-widgets/ipyleaflet
* Documentation: https://ipyleaflet.readthedocs.io

### ipyleaflet - simplest map

In [3]:
from ipyleaflet import *

m = Map(center=(-1.4563, -48.5013), zoom=10, basemap=basemaps.OpenStreetMap.Mapnik)
display(m)

### ipyleaflet - add overlay tiles
This basemap is now transparently overlayed with tiles from the [Strava heatmap](https://www.strava.com/heatmap).

In [4]:
strava_all = basemap_to_tiles(basemaps.Strava.All)
m.add_layer(strava_all)
display(m)

### ipyleaflet - mouse interaction handling
We'll remove the overlay and add interaction to show lat/lon coordinates where your mouse is. 

In [5]:
m.remove_layer(strava_all)

from ipywidgets import Label

label = Label()
display(label)

def handle_interaction(**kwargs):
    if kwargs.get('type') == 'mousemove':
        label.value = str(kwargs.get('coordinates'))

m.on_interaction(handle_interaction)
display(m)

### ipyleaflet - Add Overlay Vector Layer
We will add some rivers.


In [6]:
import json

with open('../data/rivers_lake_centerlines.json') as f:
    data = json.load(f)
    
for feature in data['features']:
    feature['properties']['style'] = {
        'color': 'blue',
        'weight': 1,
        'fillColor': 'blue',
        'fillOpacity': 0.5
    }
geo = GeoJSON(data=data, hover_style={'fillColor': 'red'}, name='Rivers-Lakes')
m.zoom = 8
m.add_layer(geo)
display(m)

### ipyleaflet - Adding Control
Add a control to select layers to display on map.


In [7]:
m.add_control(LayersControl())

### ipyleaflet - Creating two maps side by side
As ipyleaflet is based on ipywidgets, we can make interesting combinations.


In [8]:
import ipywidgets
 
ipywidgets.HBox([m, Map(center=[-1.4563, -48.5013], zoom=8)])

### ipyleaflet - Adding Popups

For this example we'll prepare a map of the following scenario
Seeing all the pubs and restaurants as a point on the map and on click show their name

In [9]:
import json
from ipyleaflet import Map, Marker, MarkerCluster, Popup
from ipywidgets import HTML

m = Map(center=(45.75, 21.25),zoom=12)

# Load GeoJSON
with open('../data/poi-timis.geojson') as f:
    data = json.load(f)

# Create markers from Point features
markers = []
for feature in data['features']:
    lon, lat = feature['geometry']['coordinates']
    pophtml = f"<b>{feature['properties'].get('name')}</b><br>{feature['properties'].get('fclass')}"
    marker = Marker(
      location=(lat, lon),
      title=feature['properties'].get('name', '')
    )
    marker.popup = HTML(value=pophtml)
    markers.append(marker)

# Create cluster layer
m.add(MarkerCluster(markers=markers, disable_clustering_at_zoom=16))
display(m)

### Other interesting ipyleaflet functions

1. AntPath 
2. Heatmap
3. Velocity
4. Choropleth

check out out at https://ipyleaflet.readthedocs.io/

## Bokeh

Bokeh is a powerful framework to produce tailored interactive map and data visualisations.
Map features are limited compared to Folium, but there are more options to tailor the behaviour.
Bokeh provides mechanisms to interact with a server side application. With Geopandas and Bokeh
one can produce a nice looking interactive map like in the image below:

![Bokeh and Geopandas Example](images/bokeh-example1.jpg)
*Interactive Map with Bokeh and GeoPandas - Source: [CSC L6](https://automating-gis-processes.github.io/CSC/lessons/L6/interactive-map-bokeh.html)*


### Bokeh - links

See also:

* https://automating-gis-processes.github.io/CSC/lessons/L6/interactive-map-bokeh.html
* [Binder for Geographic Plots in Bokeh](https://mybinder.org/v2/gh/bokeh/bokeh-notebooks/master?filepath=tutorial%2F09%20-%20Geographic%20Plots.ipynb)
* https://towardsdatascience.com/exploring-and-visualizing-chicago-transit-data-using-pandas-and-bokeh-part-ii-intro-to-bokeh-5dca6c5ced10
* https://pythonawesome.com/bokeh-plotting-backend-for-pandas-and-geopandas/


### Bokeh - making a simple plot
First, we learn the basic logic of plotting in Bokeh by making a simple interactive plot with a few points.

In [10]:
# Import the necessary functionalities from Bokeh
from bokeh.plotting import figure, save
from bokeh.io import output_notebook, show
# Initialize our plot by calling the `figure` object.
p = figure(title='My first interactive plot!')
# Next we create lists of x and y coordinates that we want to plot.
x_coords = [0,1,2,3,4]
y_coords = [5,4,1,2,0]
# Now we can plot those as points using a `.scatter()` -object. Give it a red color and size of 10.
p.scatter(x=x_coords, y=y_coords, size=10, color='red')
# direct the bokeh output to the notebook
output_notebook()
# show the output
show(p)

### Bokeh - Creating an Interactive Tiled Background Map


In [16]:
from bokeh.plotting import figure
from bokeh.io import output_notebook, show
output_notebook()
# add tile layer, bounds supplied in web mercator
p = figure(tools='pan, wheel_zoom', x_range=(-10000000, -3000000), y_range=(-6000000, 0),
           x_axis_type='mercator', y_axis_type='mercator')
p.add_tile("CartoDB Positron", retina=True)

show(p)

### Creating an Interactive Maps using Bokeh and Geopandas

Creating an interactive Bokeh map from a Shapefile or other vector data file like GeoJSON
consists typically of the following steps:

* Read the spatial vector file into `GeoDataFrame`
* Calculate the x and y coordinates of the geometries into separate columns
* Convert the `GeoDataFrame` into a Bokeh `DataSource`
* Plot the x and y coordinates as points, lines or polygons (which are in Bokeh words: `scatter`, `multi_line` and `patches`)

We follow the steps below, extending and plotting on the tiled map from above.


In [18]:
import geopandas as gpd

# Read the data (already in Web Mercator projection (ignore the warning)
points = gpd.read_file('../data/populated_places.3857.gpkg')

In [19]:
def getPointCoords(row, geom, coord_type):
    """Calculates coordinates ('x' or 'y') of a Point geometry"""
    if coord_type == 'x':
        return row[geom].x
    elif coord_type == 'y':
        return row[geom].y

In [20]:
# Set coordintates for points
points['x'] = points.apply(getPointCoords, geom='geometry', coord_type='x', axis=1)
points['y'] = points.apply(getPointCoords, geom='geometry', coord_type='y', axis=1)
# Select top 5 points
points.head(5)


In [22]:
# Remove geometry from columns 
p_df = points.drop('geometry', axis=1).copy()
# Select top 2 points
p_df.head(2)


In [23]:
# As Columns
from bokeh.models import ColumnDataSource, LabelSet
psource = ColumnDataSource(p_df)
p.scatter('x', 'y', source=psource, color='lightblue',size=15, alpha=0.7)
labels = LabelSet(x='x', y='y', text='NAME', source=psource, x_offset=5, y_offset=5)
p.add_layout(labels)
show(p)

### Using Bokeh GeoJSONSource  - OPTIONAL
The above scenario could be effected even more compact.
The Bokeh `GeoJSONDataSource` expects a GeoJSON-string, so we can 
just use ordinary file `open()/read()`.

See `GeoJSONDataSource` example in the [Bokeh Documentation](https://bokeh.pydata.org/en/latest/docs/user_guide/geo.html#geojson-data).
Though we have to do reprojection from WGS84 (default for GeoJSON)
to EPSG:3857 (Web Mercator a.k.a. "Google" or "OSM" Projection) before. In this case we have already reprojected
the GeoJSON file using `ogr2ogr`:

	ogr2ogr -s_srs EPSG:4326 -t_srs EPSG:3857 populated_places.3857.json populated_places.json


<div class="alert alert-block alert-warning">
<b>Warning:</b> Although GDAL/OGR, bokeh and many other tools support a crs member in GeoJSON, please note that the latest version of the <a href="https://datatracker.ietf.org/doc/html/rfc7946">GeoJSON spec<a> does not support specifying Coordinate Reference Systems and always assumes a default WGS84 crs. If you need crs support in GeoJSON, you consider the OGC <a href="https://github.com/opengeospatial/ogc-feat-geo-json">JSON-FG candidate Standard</a> from OGC.
</div>

In [25]:
from bokeh.models import GeoJSONDataSource
from bokeh.plotting import figure
from bokeh.io import output_notebook, show
output_notebook()

# We could also assemble map plus overlay in an HTML file
# output_file("../data/output/07-bokeh-geojson.html")

Read the populated places GeoJSON


In [26]:
with open('../data/populated_places.3857.json') as json_file:
        geojson = json_file.read()
        
geo_source = GeoJSONDataSource(geojson=geojson)

Build the map. Again, use the wheel zoom and pan tools to navigate the map. 


In [27]:
p = figure(tools='pan, wheel_zoom', x_axis_type='mercator', y_axis_type='mercator', width=800, height=500)

# add background tiles layer from CARTO
p.add_tile("CartoDB Positron", retina=True)

# add populated places point overlay
p.scatter(x='x', y='y', size=10, alpha=0.7, source=geo_source, color='lightblue', legend_label='Populated Places')

show(p)

## pydeck
pydeck is a vizualisation library based on deck.gl, which facilitates 3D vizualisations
The example below, shows the distribution of hotels, restaurants and pubs in Timisoara as hexagons.

In [16]:
import json
import pydeck as pdk

# Load GeoJSON
with open("../data/poi-timis.geojson") as f:
    geojson = json.load(f)

# Extract coordinates for HexagonLayer
points = [
    feature["geometry"]["coordinates"]
    for feature in geojson["features"]
    if feature["geometry"]["type"] == "Point"
]

hex_layer = pdk.Layer(
    "HexagonLayer",
    data=points,
    get_position="-",
    radius=100,          # meters
    elevation_scale=5,  # height scaling
    elevation_range=[0, 200],
    extruded=True,
    pickable=True,
    auto_highlight=True,
    coverage=1,
    color_range=[
        [255, 255, 204],  # low density
        [199, 233, 180],
        [127, 205, 187],
        [65, 182, 196],
        [44, 127, 184],
        [37, 52, 148],    # high density (dark)
    ],
)

view_state = pdk.ViewState(
    latitude=45.75,
    longitude=21.25,
    zoom=12,
    pitch=45,
)

deck = pdk.Deck(
    layers=[hex_layer],
    initial_view_state=view_state,
    map_style="light",
    tooltip={
        "html": "<b>Points:</b> {elevationValue}",
    },
)

deck

---
[<- Data analysis](06-data-analysis.ipynb) | [Metadata ->](08-metadata.ipynb)